## Data Cleaning

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/pricing_analyst_dataset.csv")

df_raw = pd.read_csv(DATA_PATH)

display(df_raw.head())
print(df_raw.shape)
df_raw.info()

,offerdate,sold_premium,offered_premium,purchase_price,purchase_date,item age,pricing_point,predictedconversionrate,plan_flag,plan_count,...,planscancelled_lastyear_count,claims_count,claim_amount,price_diff,IsModel,sale_flag,base_rate,manufacturerbrandname_enc,itemcategoryname_enc,itemsupercategorycode_enc
0,17/03/2023,NaN,32.64,89.99,16/03/2023,1,@22%,0.15,0,0.0,...,0.0,0.0,0.00,-0.058824,Yes,0,34.68,56,35,14
1,01/03/2023,69.72,69.72,329.00,24/12/2022,67,@22%,0.81,1,5.0,...,2.0,4.0,347.12,-0.023529,Yes,1,71.40,123,34,3
2,12/04/2023,NaN,48.24,249.00,05/04/2023,7,@23%,0.08,0,0.0,...,0.0,0.0,0.00,0.210843,Yes,0,39.84,7,16,12
3,09/03/2023,NaN,91.92,746.42,09/03/2021,730,@23%,0.32,0,0.0,...,0.0,0.0,0.00,0.298305,Yes,0,70.80,107,36,4
4,18/03/2023,NaN,89.64,493.98,18/03/2023,0,@22%,0.25,1,1.0,...,0.0,0.0,0.00,0.299130,Yes,0,69.00,57,36,4


(8867, 21)
<class 'pandas.DataFrame'>
RangeIndex: 8867 entries, 0 to 8866
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   offerdate                      8867 non-null   str    
 1   sold_premium                   1991 non-null   float64
 2   offered_premium                8867 non-null   float64
 3   purchase_price                 8866 non-null   float64
 4   purchase_date                  8867 non-null   str    
 5   item age                       8867 non-null   str    
 6   pricing_point                  8867 non-null   str    
 7   predictedconversionrate        8867 non-null   float64
 8   plan_flag                      8867 non-null   int64  
 9   plan_count                     8826 non-null   float64
 10  plansactive_lastyear_count     8826 non-null   float64
 11  planscancelled_lastyear_count  8826 non-null   float64
 12  claims_count                   7718 non-null   f

In [4]:
quality_check = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "missing_count": df_raw.isna().sum().values,
    "missing_pct": (df_raw.isna().mean() * 100).round(2).values,
    "unique_values": df_raw.nunique(dropna=True).values
})

display(quality_check.sort_values("missing_pct", ascending=False))
print(f"Duplicate rows: {df_raw.duplicated().sum():,}")

for col in ["pricingpoint", "saleflag", "IsModel", "planflag"]:
    if col in df_raw.columns:
        print(f"\n{col}")
        display(df_raw[col].value_counts(dropna=False))

,column,dtype,missing_count,missing_pct,unique_values
1,sold_premium,float64,6876,77.55,193
13,claim_amount,float64,1149,12.96,435
12,claims_count,float64,1149,12.96,14
11,planscancelled_lastyear_count,float64,41,0.46,15
9,plan_count,float64,41,0.46,6
10,plansactive_lastyear_count,float64,41,0.46,27
3,purchase_price,float64,1,0.01,1270
0,offerdate,str,0,0.00,52
2,offered_premium,float64,0,0.00,300
4,purchase_date,str,0,0.00,571


Duplicate rows: 2

IsModel


IsModel
Yes    7750
No     1117
Name: count, dtype: int64

In [7]:
df = df_raw.copy()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

date_columns = ["offerdate", "purchase_date"]
numeric_columns = [
    "sold_premium",
    "offered_premium",
    "purchase_price",
    "item_age",
    "predictedconversionrate",
    "plan_count",
    "plansactive_lastyear_count",
    "planscancelled_lastyear_count",
    "claims_count",
    "claim_amount",
    "price_diff",
    "base_rate"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["sale_flag"] = pd.to_numeric(df["sale_flag"], errors="coerce")
df["pricing_point"] = df["pricing_point"].astype("string").str.strip()
df["ismodel"] = df["ismodel"].astype("string").str.strip().str.lower()
df["plan_flag"] = df["plan_flag"].astype("string").str.strip().str.lower()

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8867 entries, 0 to 8866
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   offerdate                      8867 non-null   datetime64[us]
 1   sold_premium                   1991 non-null   float64       
 2   offered_premium                8867 non-null   float64       
 3   purchase_price                 8866 non-null   float64       
 4   purchase_date                  8867 non-null   datetime64[us]
 5   item_age                       8864 non-null   float64       
 6   pricing_point                  8867 non-null   string        
 7   predictedconversionrate        8867 non-null   float64       
 8   plan_flag                      8867 non-null   string        
 9   plan_count                     8826 non-null   float64       
 10  plansactive_lastyear_count     8826 non-null   float64       
 11  planscancelled_lastyear_coun

In [15]:
required_columns = [
    "pricing_point",
    "sale_flag",
    "base_rate",
    "offered_premium",
    "price_diff"
]

df["invalid_price_flag"] = (
    (df["base_rate"] <= 0)
    | (df["offered_premium"] <= 0)
    | (df["sold_premium"].notna() & (df["sold_premium"] <= 0))
)

df["price_diff_check"] = (
    df["offered_premium"] - df["base_rate"] - df["price_diff"]
)

df["price_diff_matches"] = np.isclose(
    df["price_diff_check"],
    0,
    atol=0.01,
    equal_nan=False
)

df_analysis = (
    df.drop_duplicates()
      .dropna(subset=required_columns)
      .query("invalid_price_flag == False")
      .copy()
)

print(f"Original rows: {len(df):,}")
print(f"Analysis-ready rows: {len(df_analysis):,}")
print(f"Rows removed: {len(df) - len(df_analysis):,}")

display(
    df_analysis[
        ["pricing_point", "sale_flag", "base_rate", "offered_premium", "price_diff", "price_diff_matches"]
    ].head()
)

Original rows: 8,867
Analysis-ready rows: 8,865
Rows removed: 2


,pricing_point,sale_flag,base_rate,offered_premium,price_diff,price_diff_matches
0,@22%,0,34.68,32.64,-0.058824,False
1,@22%,1,71.40,69.72,-0.023529,False
2,@23%,0,39.84,48.24,0.210843,False
3,@23%,0,70.80,91.92,0.298305,False
4,@22%,0,69.00,89.64,0.299130,False


In [17]:
df["invalid_price_flag"].value_counts(False)

invalid_price_flag
False    8867
Name: count, dtype: int64

In [18]:
from pathlib import Path

PROCESSED_DATA_DIR = Path("../data")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DATA_DIR / "pricing_analyst_cleaned.csv"

df_analysis.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"Rows saved: {len(df_analysis):,}")
print(f"Columns saved: {df_analysis.shape[1]:,}")

Cleaned dataset saved to: ..\data\pricing_analyst_cleaned.csv
Rows saved: 8,865
Columns saved: 24
